# RAG Workshop: Build Your Own Question-Answering AI

Welcome. You will build an AI that can answer questions about any document you give it. No coding experience needed.

**What you will do**

1. Set up the system (one button)
2. Ask questions about a sample document
3. Change how the AI "thinks" by editing its instructions
4. Upload your own document and ask it questions
5. Experiment with how the AI finds information
6. Watch the AI hallucinate, then fix it

**Before you start:** Get a free Google AI API key here (takes 30 seconds, no credit card): https://aistudio.google.com/apikey

Then run the cell below.

In [ ]:
#@title Run me first (takes about 60 seconds) { display-mode: "form" }

# 1. INSTALL TOOLS: Think of these as the 'apps' our notebook needs to work.
!pip install -q google-genai chromadb sentence-transformers pypdf 2>/dev/null

import getpass
import io
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader

# 2. CONNECT TO THE BRAIN: We link this notebook to Google's Gemini AI.
print("Paste your Google AI API key (get one at https://aistudio.google.com/apikey)")
GOOGLE_API_KEY = getpass.getpass("API key: ")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# 3. LOAD THE TRANSLATOR: This model turns human words into a list of numbers (embeddings).
# Computers find information much faster by comparing numbers than by comparing words.
print("\nLoading the embedding model... (one-time, about 30 seconds)")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model ready.")

# 4. CREATE THE FILING CABINET: A 'Vector Database' is where we store those number-versions of our text.
chroma_client = chromadb.Client()

# === Helper functions: The 'gears' inside the machine ===

import re

# --- CHUNKING STRATEGIES ---------------------------------------------------
# Chunking is the FIRST thing that happens to your document, and the most
# important. Retrieval can only ever find what chunking kept intact. Each
# strategy below splits text differently, and each fails in its own way.

def chunk_fixed(text, chunk_size=500, overlap=50):
    """Blind character windows. Fast, but happily cuts words and sentences in half."""
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

def chunk_sentence(text, sentences_per_chunk=4):
    """Group whole sentences together. Never splits mid-thought,
    but ignores where one topic ends and the next begins."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    out = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i + sentences_per_chunk]).strip()
        if chunk:
            out.append(chunk)
    return out

def chunk_paragraph(text):
    """Split on blank lines. Respects how the author organized the document.
    Great for clean docs like this handbook, weak on a wall-of-text PDF."""
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

def chunk_recursive(text, chunk_size=500):
    """Keep paragraphs whole; if one is too big, fall back to sentences.
    This is basically what real frameworks (LangChain's
    RecursiveCharacterTextSplitter) do under the hood."""
    out = []
    for para in (p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()):
        if len(para) <= chunk_size:
            out.append(para)
            continue
        buf = ""
        for s in re.split(r'(?<=[.!?])\s+', para):
            if len(buf) + len(s) <= chunk_size:
                buf = (buf + " " + s).strip()
            else:
                if buf:
                    out.append(buf)
                buf = s
        if buf:
            out.append(buf)
    return [c for c in out if c]

def chunk_text(text, strategy="Fixed size", chunk_size=500, overlap=50):
    """Pick a chunking strategy. This is the dial that matters most."""
    if strategy == "Fixed size":
        return chunk_fixed(text, chunk_size, overlap)
    if strategy == "By sentence":
        return chunk_sentence(text)
    if strategy == "By paragraph":
        return chunk_paragraph(text)
    if strategy == "Smart (recursive)":
        return chunk_recursive(text, chunk_size)
    raise ValueError(f"Unknown strategy: {strategy}")

# --- INDEXING --------------------------------------------------------------

def load_document(text, source_name="document", strategy="Fixed size",
                  chunk_size=500, overlap=50):
    """INDEXING: chop the text, translate each chunk to numbers, and file it
    away. We remember the raw text so other cells can re-chop it later."""
    global collection, current_text, current_source, current_strategy
    current_text, current_source, current_strategy = text, source_name, strategy
    try:
        chroma_client.delete_collection(name="knowledge")
    except Exception:
        pass
    collection = chroma_client.create_collection(name="knowledge")
    chunks = chunk_text(text, strategy, chunk_size, overlap)
    embeddings = embedding_model.encode(chunks, show_progress_bar=False).tolist()
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
    )
    print(f"Loaded {len(chunks)} chunks from: {source_name}  (strategy: {strategy})")

# --- RETRIEVAL & GENERATION ------------------------------------------------

def retrieve(question, top_k=3):
    """Turn the question into numbers and find the closest chunks.
    Also returns a 'distance' for each: lower means more relevant."""
    q_embedding = embedding_model.encode(question, show_progress_bar=False).tolist()
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=["documents", "distances"],
    )
    return results["documents"][0], results["distances"][0]

def answer_question(question, top_k=3,
                    system_prompt="You are a helpful expert. Answer using the context below."):
    """Quiet version: retrieve, ask the model, RETURN the answer. No printing.
    Used by the scoring cell so it can grade many questions at once."""
    chunks, distances = retrieve(question, top_k)
    context = "\n\n---\n\n".join(chunks)
    prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text, chunks, distances

def ask(question, top_k=3,
        system_prompt="You are a helpful expert. Answer using the context below."):
    """The pretty version: shows the question, the retrieved chunks (with how
    relevant each one was), and the AI's answer."""
    answer, chunks, distances = answer_question(question, top_k, system_prompt)

    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)
    print(f"\nRETRIEVED {len(chunks)} CHUNK(S) FROM THE DOCUMENT:")
    for i, (chunk, dist) in enumerate(zip(chunks, distances), 1):
        preview = chunk[:250] + ("..." if len(chunk) > 250 else "")
        print(f"\n  [Chunk {i}]  (distance {dist:.3f} - lower is more relevant)\n  {preview}")

    print("\n" + "=" * 60 + "\nAI ANSWER:\n" + "=" * 60)
    print(answer)

# Load the sample document
SAMPLE_DOC = """The Builder's Handbook: Shipping AI Products from Zero to One

What is RAG?

RAG stands for Retrieval Augmented Generation. It is a technique that lets large language models answer questions using information they were not originally trained on. Instead of relying only on the model's internal knowledge, RAG retrieves relevant pieces of information from a knowledge base and gives them to the model as context. This makes the answers more accurate, more current, and grounded in sources you control.

The classic problem RAG solves: language models hallucinate. They make up plausible-sounding facts. RAG reduces hallucination by forcing the model to base its answer on documents you actually trust.

The Four Pieces of a RAG Pipeline

Every RAG system has four core parts. First, a chunker that splits long documents into smaller pieces. Second, an embedding model that turns each chunk into a list of numbers representing its meaning. Third, a vector database that stores these embeddings and lets you search them quickly. Fourth, a large language model that generates the final answer using the retrieved chunks as context. Each piece can be swapped independently, which is what makes RAG flexible.

Choosing a Vector Database

There are many vector databases now. The most popular for beginners is ChromaDB. It runs in memory, requires no setup, and is perfect for prototypes and small projects. Pinecone and Weaviate are cloud-hosted options that scale better but cost money. Qdrant and LanceDB are open source alternatives that you can self-host. For a workshop or personal project, ChromaDB is the right choice. You only think about scaling once you have real users and real load.

Common Mistakes New Builders Make

The first mistake is overcomplicating the stack. New builders often reach for LangChain or LlamaIndex on day one. These frameworks are powerful but they hide the mechanics. You learn faster by writing the pipeline yourself first.

The second mistake is ignoring chunk size. Too-large chunks lose precision. Too-small chunks lose context. A good starting point is 500 characters with 50 character overlap.

The third mistake is skipping evaluation. Without a small test set of questions and expected answers, you have no idea if your changes are improving or hurting the system. Even ten test questions is enough to catch most regressions.

The fourth mistake is trusting the model too much. Always show users the sources behind an answer. Always include a way for the model to say it does not know. Models will confidently invent answers if you let them.

Prompt Engineering Basics

The system prompt is where you set the personality and constraints of the AI. Good system prompts are specific. Instead of "be helpful," say "answer in two sentences using only the context provided. If the context does not contain the answer, say I do not know." Specificity beats cleverness. The shortest prompt that gets the right behavior is the best prompt. Long elaborate prompts often confuse the model and produce worse outputs.

Tips for Non-Technical Founders

You do not need to write code to build with AI. The most important skill is understanding what AI can and cannot do, and designing products around its real capabilities. Start with the problem, not the technology. Many founders see a cool model and try to invent a product around it. The successful path is the opposite: find a problem people will pay to solve, then ask whether AI is the right tool.

Use no-code and low-code tools to prototype before hiring engineers. Bubble, Glide, Make, and Zapier can build surprisingly capable AI products without a single line of code. The MVP exists to test the hypothesis, not to win design awards.

Erasmus AI Builders Community

The Erasmus AI Builders Committee runs a recurring meetup series focused on practical AI tooling and agent development. Past sessions have covered Claude Code, Codex, and agent plugin architectures. Workshops are open to students from any faculty and prior coding experience is not required. The community emphasizes shipping over studying. The motto: build something every week, even if it is small. Members regularly demo personal projects and trade feedback on prompts, architectures, and product positioning.

Resources for Going Deeper

For learning RAG specifically, the LlamaIndex documentation has the best free walkthroughs. For prompt engineering, the Anthropic prompt engineering guide is comprehensive and free. For staying current on the model landscape, the LocalLLaMA subreddit moves faster than most newsletters. Two books worth reading: Designing Machine Learning Systems by Chip Huyen, and AI Engineering by the same author."""

load_document(SAMPLE_DOC, "The Builder's Handbook (sample)")
print("\nAll set. Move to the next cell.")

## Step 1: Ask your first question

The sample document is a short Builder's Handbook for AI founders. Type any question in the box below and press play. You will see:

- The question you asked
- Which parts of the document the AI found relevant
- The AI's answer

**Try these:**
- What is RAG?
- How do I choose a vector database?
- What mistakes do new builders make?
- What is the Erasmus AI community about?
- Should I use LangChain?

In [ ]:
#@title Ask the sample document { display-mode: "form" }
question = "What is RAG?" #@param {type:"string"}
ask(question)


## Step 2: Change the AI's personality

The "system prompt" tells the AI how to behave. Same question, same document, very different answers depending on the instructions you give.

Pick a personality from the dropdown, run the cell, then try the same question with a different personality. Notice how the answer changes but the retrieved chunks stay the same. The retrieval did not change, only the way the AI talks about what it found.

In [ ]:
#@title Try different personalities { display-mode: "form" }
question = "How do I get better at building AI products?" #@param {type:"string"}
personality = "Explain to a 5 year old" #@param ["Helpful expert", "Pirate captain", "Explain to a 5 year old", "Skeptical scientist", "One sentence only"]

personalities = {
    "Helpful expert": "You are a helpful expert. Answer clearly using the context below.",
    "Pirate captain": "You are a pirate captain. Answer in pirate speak using the context below. Arrr!",
    "Explain to a 5 year old": "Explain the answer like the reader is 5 years old. Use simple words and short sentences.",
    "Skeptical scientist": "You are a skeptical scientist. Answer cautiously and point out anything uncertain in the context.",
    "One sentence only": "Answer in exactly one sentence using the context below.",
}

ask(question, system_prompt=personalities[personality])


## Step 3: Upload your own document

Now the fun part. Upload any PDF or text file. The AI will be able to answer questions about it instead of the sample document.

**Good things to try:**
- A research paper you are reading
- Your course syllabus or lecture notes
- A book chapter
- A long article you saved
- A company's terms of service (then ask scary questions)

Run the cell, click "Choose Files" when prompted, and pick a file from your computer.

In [ ]:
#@title Upload a PDF or text file
from google.colab import files
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(io.BytesIO(content))
        text = "\n\n".join([(page.extract_text() or "") for page in reader.pages])
    else:
        text = content.decode("utf-8", errors="ignore")
    load_document(text, filename)
    print(f"\nYou can now ask questions about: {filename}")


Now ask your document anything. Type your question and press play.

In [ ]:
#@title Ask about your uploaded document { display-mode: "form" }
question = "What is this document about?" #@param {type:"string"}
ask(question)


## Step 4: How much should the AI read?

The AI does not read your whole document every time. It finds the most relevant chunks first, then uses them to answer. The slider controls how many chunks it looks at.

- **Low (1-2):** Fast, focused, but might miss context
- **Medium (3-5):** Usually best
- **High (8-10):** More context but also more noise, can confuse the model

Try the same question at different settings and see how the answer changes.

In [ ]:
#@title Adjust how many chunks the AI looks at { display-mode: "form" }
question = "What are common mistakes new builders make?" #@param {type:"string"}
chunks_to_retrieve = 4 #@param {type:"slider", min:1, max:10, step:1}
ask(question, top_k=chunks_to_retrieve)


## Step 5: Change how the document gets chopped up

This is the most important dial in the whole system, and until now it has been
hidden. Before anything else happens, your document is **chunked** - sliced into
small pieces. Retrieval can only ever find what chunking kept intact. If a fact
got split across two chunks, no amount of clever prompting will recover it.

The original chunker used blind character windows: it would happily slice a word
or a sentence in half. Below you can swap between four strategies and *see* the
difference.

- **Fixed size** - cut every N characters. Fast, but cuts mid-word.
- **By sentence** - never splits a sentence, but ignores topic shifts.
- **By paragraph** - respects the document's structure (great for this handbook).
- **Smart (recursive)** - keeps paragraphs whole, falls back to sentences when
  one is too big. This is what production frameworks do.

In [ ]:
#@title See how each strategy chops the document { display-mode: "form" }
strategy = "Fixed size" #@param ["Fixed size", "By sentence", "By paragraph", "Smart (recursive)"]
chunk_size = 500 #@param {type:"slider", min:100, max:1000, step:50}

preview_chunks = chunk_text(current_text, strategy=strategy, chunk_size=chunk_size)
print(f"Strategy: {strategy}  ->  {len(preview_chunks)} chunks\n")
for i, c in enumerate(preview_chunks[:6], 1):
    preview = c[:200] + ("..." if len(c) > 200 else "")
    print(f"[Chunk {i}]  ({len(c)} chars)\n{preview}\n")
if len(preview_chunks) > 6:
    print(f"...and {len(preview_chunks) - 6} more chunks not shown.")

In [ ]:
#@title Re-chop the document with a strategy, then ask the same question { display-mode: "form" }
strategy = "Smart (recursive)" #@param ["Fixed size", "By sentence", "By paragraph", "Smart (recursive)"]
chunk_size = 500 #@param {type:"slider", min:100, max:1000, step:50}
question = "What chunk size does the handbook recommend?" #@param {type:"string"}

# Re-index whatever document is currently loaded, using the chosen strategy.
load_document(current_text, current_source, strategy=strategy, chunk_size=chunk_size)
ask(question)

## Step 6: Watch the AI hallucinate, then stop it

This is the most important lesson of the workshop.

Ask a question whose answer is NOT in the document. The default AI will often invent a confident answer. This is hallucination.

Try this question first with `include_safeguard` set to **False**, then set it to **True** and run again. See the difference.

**Questions to try (none of these are in the sample document):**
- Who invented the printing press?
- What is the capital of Brazil?
- How do I bake sourdough bread?
- What year did the Roman Empire fall?

In [ ]:
#@title Compare with and without a safeguard prompt { display-mode: "form" }
question = "Who invented the printing press?" #@param {type:"string"}
include_safeguard = False #@param {type:"boolean"}

base_prompt = "You are a helpful expert. Answer using the context below."
safeguard = " Only answer if the context clearly contains the answer. If it does not, say exactly: 'I cannot find this in the document.'"

prompt = base_prompt + (safeguard if include_safeguard else "")
ask(question, system_prompt=prompt)


## Step 7: Score your whole system

So far, "is this better?" has been a gut feeling. Real builders don't guess -
they measure. The handbook even warns that skipping evaluation is a top mistake.

Below is a tiny **test set**: 5 questions about the sample handbook, each with
the fact the answer must contain. The cell reloads the sample document, runs all
5 questions, and gives you a score.

Now you have a game: **tune the dials until you hit 5/5.** Try different chunking
strategies, chunk sizes, and numbers of retrieved chunks. When the score moves,
ask yourself *why* - that "why" is the whole skill of building RAG.

In [ ]:
#@title Score the system on a test set { display-mode: "form" }
#@markdown This reloads the **sample handbook** and grades it. Tune the dials to hit 5/5.
strategy = "Smart (recursive)" #@param ["Fixed size", "By sentence", "By paragraph", "Smart (recursive)"]
chunk_size = 500 #@param {type:"slider", min:100, max:1000, step:50}
chunks_to_retrieve = 3 #@param {type:"slider", min:1, max:10, step:1}

TEST_SET = [
    {"q": "What does RAG stand for?",
     "any_of": ["retrieval augmented generation", "retrieval-augmented generation"]},
    {"q": "What chunk size does the handbook recommend as a starting point?",
     "any_of": ["500"]},
    {"q": "Which vector database does the handbook recommend for beginners?",
     "any_of": ["chroma"]},
    {"q": "What is the motto of the Erasmus AI Builders community?",
     "any_of": ["build something every week"]},
    {"q": "Name one common mistake new builders make.",
     "any_of": ["overcomplicat", "langchain", "llamaindex", "chunk size",
                "evaluation", "trusting the model", "trust the model"]},
]

load_document(SAMPLE_DOC, "The Builder's Handbook (sample)",
              strategy=strategy, chunk_size=chunk_size)

passed = 0
print("=" * 60)
for item in TEST_SET:
    answer, _, _ = answer_question(item["q"], top_k=chunks_to_retrieve)
    ok = any(kw.lower() in answer.lower() for kw in item["any_of"])
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {item['q']}")
    if not ok:
        print(f"        wanted one of: {item['any_of']}")
        print(f"        got: {answer.strip()[:120]}...")
print("=" * 60)
print(f"SCORE: {passed}/{len(TEST_SET)}  ({100 * passed // len(TEST_SET)}%)")
if passed == len(TEST_SET):
    print("\nPerfect score. Now break it: which strategy drops you below 5/5, and why?")
else:
    print("\nNot 5/5 yet. Change the strategy / chunk size / number of chunks above and re-run.")

## You did it

You just built a retrieval-augmented question answering system. Behind the scenes it:

1. Split your document into small chunks
2. Turned each chunk into a list of numbers (an embedding) representing its meaning
3. Stored those numbers in a vector database
4. When you asked a question, turned the question into numbers too
5. Found the chunks whose numbers were closest to your question's numbers
6. Sent those chunks to the AI as context, along with your question
7. Returned the AI's answer

That is RAG. Everything else (chunk sizes, embedding models, vector databases, prompt design) is just turning these dials to make the system work better.


## From toy to tool: building for real

What you built is a great *learning* system, but it is a toy: the database lives
in memory and disappears when this tab closes, it serves one file, and it only
runs while you babysit the notebook. Here is the ladder from here to a real
product. Each rung is a small, concrete change.

**1. Make the knowledge base survive a restart.** Change one line:
`chromadb.Client()` -> `chromadb.PersistentClient(path="./rag_db")`. Now your
index is written to disk. (Demo below.)

**2. Give it a real interface.** A notebook is not a product. Wrap the pipeline
in a chat app - the cell below launches one with a public link, right from Colab.

**3. Ship it.** Deploy that app to a free [Hugging Face Space](https://huggingface.co/spaces),
share the link, and demo it at a meetup. This is the "build something every week"
move.

**4. Now reach for frameworks.** You understand the four pieces by hand, so
LangChain and LlamaIndex stop being magic and become shortcuts. That is the right
moment to pick them up - not before.

In [ ]:
#@title (Optional) Make your knowledge base survive a restart { display-mode: "form" }
# The ONLY real change from the toy version is PersistentClient instead of Client.
# The index is now written to ./rag_db and survives restarting the notebook.
persistent_client = chromadb.PersistentClient(path="./rag_db")
try:
    persistent_client.delete_collection("knowledge")
except Exception:
    pass
pcol = persistent_client.create_collection("knowledge")
chunks = chunk_text(current_text, strategy=current_strategy)
pcol.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embedding_model.encode(chunks, show_progress_bar=False).tolist(),
    documents=chunks,
)
print(f"Saved {len(chunks)} chunks to ./rag_db - this folder survives restarts.")

In [ ]:
#@title Turn your RAG system into a chat app { display-mode: "form" }
!pip install -q gradio 2>/dev/null
import gradio as gr

def chat_fn(question, history):
    answer, chunks, distances = answer_question(question, top_k=3)
    sources = "\n\n".join(f"- {c[:200]}..." for c in chunks)
    return f"{answer}\n\n---\n**Sources used:**\n{sources}"

gr.ChatInterface(
    chat_fn,
    title="Chat with your document",
    description=f"Ask questions about: {current_source}",
).launch(share=True)
# A public 'share' link will appear above. It stays live while this cell runs.

## Build something this week

Pick one. Each is a real, finishable project using exactly what you learned:

- **Chat with your syllabus** - load your course outline and lecture notes, ask
  it "when is the midterm?" or "what's the grading breakdown?"
- **Study-group notes bot** - combine everyone's notes into one knowledge base
  and let the group query it. (Modify `load_document` to *add* instead of
  replace, so multiple docs live together.)
- **Interrogate the fine print** - load a company's terms of service or a rental
  contract and ask the scary questions.
- **Community docs bot** - point it at the Erasmus AI Builders notes and turn it
  into a Discord or Slack bot.

Two things separate a demo from a product: **show your sources** (so people trust
the answer) and **keep a test set** (so you know when a change helped). You now
have both.

## Keep building

Join our future meet-ups. We meet to study and ship - build something every week,
even if it is small.